# Deploy the same model to real-time, serverless, async, and batch-transform endpoints and pick the right type for three workload shapes

A product manager has a recommendation model that runs 100 inferences per second in business hours, zero overnight, and occasional 50K-prediction backfill jobs once a week. The four SageMaker endpoint types do not all fit the three workload shapes. Real-time at 100 rps with overnight idle bleeds money on a persistent instance. Serverless handles the bursty business-hours load with $0 idle but cold-start can bite at low rps. Async fits long-running predictions or callback-style use cases. Batch Transform is the right tool for the weekly 50K backfill.

You have one session to stand all four up against the SAME model, measure cost-per-1K-predictions and p95 latency for each, then write the recommendation. The cost trap: the real-time and async endpoints on ml.t2.medium bill $0.07/hour continuously while InService; cleanup tears them down in critical-first order so the residual cost is bounded.

Five artifacts ship in this lab:

- A small deterministic XGBoost model trained once for use across all four endpoint types.
- Architecture A: a Real-Time endpoint on ml.t2.medium with 100 sequential invocations measuring p95 latency.
- Architecture B: a Serverless Inference endpoint with `MemorySizeInMB=2048, MaxConcurrency=20`, with separate cold-start and warm p95 latencies.
- Architecture C: an Async Inference endpoint on ml.t2.medium that accepts an S3 input URI and writes the prediction to S3.
- Architecture D: a Batch Transform job on ml.m5.large processing 50K predictions against `s3://{bucket}/batch-input/`.

**Time.** Plan for about 70 minutes hands-on. Training is ~5 minutes. Real-time endpoint creation is ~5 minutes. Serverless is ~90 seconds. Async is ~5 minutes. Batch Transform on 50K predictions is ~10 minutes. The 100-invocation latency probes against the real-time and serverless endpoints add another 5 minutes total. Budget 120 minutes if the async output path fights back.

**Cost.** This lab costs about eight to fifteen cents if cleanup tears the real-time and async endpoints within 30 minutes of creation. Real-time on ml.t2.medium is $0.07/hour, async same, batch on ml.m5.large is $0.115/hour for ~10 minutes. Serverless and the invocations are fractions of a penny. The four endpoint types in one lab is the most-stood-up SageMaker resource count in the MLA track; the cleanup discipline matters. If you walk away with both hourly endpoints running for a day, you spend about $3.40.

This lab maps to AWS MLA-C01 Domain 3 (Deployment and Orchestration of ML Workflows, 22%) on choosing deployment infrastructure across real-time, serverless, asynchronous, batch transform, multi-model, and multi-container endpoints under cost and latency constraints.

**Compare-it lab.** Four endpoint types in one lab. Checkpoints validate that all four are stood up and inferrable; comparative reasoning (which type fits which workload shape) lives in the reflection MCQs.


In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus the SageMaker SDK for image URI lookups.

!pip install --quiet labrun-checks==0.6.0 sagemaker==2.232.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern; -rt, -sl, -async, -batch suffixes
# disambiguate the four architectures per blueprint Section 21.

import atexit
import csv
import getpass
import io
import json
import random
import sys
import statistics
import time

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "endpoint-type-comparison"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

# Resource names. Bucket and ARNs are account-suffixed and resolved after STS returns.
ENDPOINT_ROLE_NAME = f"labrun-{LAB_ID}-role"
INLINE_POLICY_NAME = "labrun-mla-lab08-role-policy"
TRAINING_JOB = f"labrun-{LAB_ID}-train"
MODEL_NAME = f"labrun-{LAB_ID}-model"
RT_CONFIG = f"labrun-{LAB_ID}-rt-config"
RT_ENDPOINT = f"labrun-{LAB_ID}-rt"
SL_CONFIG = f"labrun-{LAB_ID}-sl-config"
SL_ENDPOINT = f"labrun-{LAB_ID}-sl"
ASYNC_CONFIG = f"labrun-{LAB_ID}-async-config"
ASYNC_ENDPOINT = f"labrun-{LAB_ID}-async"
BATCH_JOB = f"labrun-{LAB_ID}-batch"

BUCKET_NAME = None
ACCOUNT_ID = None
ENDPOINT_ROLE_ARN = None
XGBOOST_IMAGE_URI = None
MODEL_ARTIFACT = None

# Captured metrics used by Checkpoint 5.
RT_P95_MS = None
RT_COST_PER_1K = None
SL_COLD_P95_MS = None
SL_WARM_P95_MS = None
SL_COST_PER_1K = None
ASYNC_LATENCY_S = None
ASYNC_COST_PER_1K = None
BATCH_WALLCLOCK_MIN = None
BATCH_COST_PER_1K = None

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# resolve account-derived names.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"MLA Batch 2 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
ENDPOINT_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{ENDPOINT_ROLE_NAME}"
print(f"Bucket: {BUCKET_NAME}")
print(f"Endpoint role ARN: {ENDPOINT_ROLE_ARN}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, orphan scan, and one-shot training.
# Manifest is module-level. labrun-checks 0.6.0 lacks adapters for SageMaker
# endpoints, configs, models, training jobs, and transform jobs. The cleanup
# cell tears those down manually BEFORE run_cleanup walks the manifest. The
# manifest below contains only adapter-supported types.
#
# Critical resources: the Real-Time endpoint and the Async endpoint, each
# billing ml.t2.medium continuously while InService at $0.07/hour. The
# Serverless endpoint is $0 idle. Cleanup tears the hourly endpoints down
# first per RESOURCE-SAFETY-SPEC Rule 2.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="iam_role",
        id=ENDPOINT_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {ENDPOINT_ROLE_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list:
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first, or remove")
    print("them manually with the AWS CLI commands the cleanup cell prints.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Train the shared model, deploy it to a Real-Time endpoint, measure p95 latency

The four architectures all use the same model. Train it once via the XGBoost built-in algorithm against a deterministic 2000-row binary-classification set, then deploy to a Real-Time endpoint on ml.t2.medium (the smallest persistent SageMaker instance).

Build it in this order:

1. Create the bucket and the endpoint role with `sagemaker.amazonaws.com` trust and an inline policy granting S3 read/write.
2. Generate a deterministic 2000-row CSV (same shape as Lab 7's, 12 features, label in column 0), 70/30 train/validation split, upload.
3. Submit one XGBoost training job; wait for `Completed` (~5 minutes); capture `MODEL_ARTIFACT`.
4. Build a SageMaker Model from the training artifact.
5. Create a Real-Time endpoint config with `ProductionVariants=[{"InstanceType": "ml.t2.medium", "InitialInstanceCount": 1, ...}]`.
6. Create the endpoint and wait for `EndpointStatus=InService` (~5 minutes).
7. Run 100 sequential invocations; warm-up excludes the first 5; capture p95 latency in ms.

ml.t2.medium is the smallest persistent SageMaker instance type at $0.07/hour. The lab uses it specifically because it makes the real-time cost-vs-serverless tradeoff visible in the cost-per-1K math.

In [ ]:
# NBVAL_SKIP
# Task 1: Train the shared model, deploy to Real-Time, measure p95 latency.

import sagemaker

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
sm = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
sm_runtime = boto3.client(
    "sagemaker-runtime",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Create the bucket with s3.create_bucket(Bucket=BUCKET_NAME).

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")


def generate_xgboost_csv(n_rows: int, seed: int = 42) -> bytes:
    rng = random.Random(seed)
    buf = io.StringIO()
    writer = csv.writer(buf)
    for _ in range(n_rows):
        features = [round(rng.gauss(0, 1), 4) for _ in range(12)]
        logit = 0.6 * features[0] - 0.4 * features[1] + 0.3 * features[2] + rng.gauss(0, 0.3)
        label = 1 if logit > 0 else 0
        writer.writerow([label] + features)
    return buf.getvalue().encode("utf-8")


full = generate_xgboost_csv(2000)
all_lines = full.splitlines(keepends=True)
split = int(len(all_lines) * 0.7)
train_bytes = b"".join(all_lines[:split])
val_bytes = b"".join(all_lines[split:])

s3.put_object(Bucket=BUCKET_NAME, Key="input/train.csv", Body=train_bytes)
s3.put_object(Bucket=BUCKET_NAME, Key="input/validation.csv", Body=val_bytes)
print(f"Uploaded train ({len(all_lines[:split])} rows) and validation ({len(all_lines[split:])} rows).")

# Also generate the 50K test set used by Async (1 prompt) and Batch Transform.
test_csv = generate_xgboost_csv(50000, seed=99)
# Strip the label column (Async and Batch expect features only).
features_only_lines = []
for line in test_csv.splitlines():
    parts = line.decode("utf-8").split(",")
    features_only_lines.append(",".join(parts[1:]))
features_only_bytes = ("\n".join(features_only_lines) + "\n").encode("utf-8")
s3.put_object(Bucket=BUCKET_NAME, Key="input/test_50k.csv", Body=features_only_bytes)
print(f"Uploaded test_50k.csv (50000 rows of features only).")

# Single-row async prompt: one feature vector.
async_prompt = features_only_lines[0].encode("utf-8")
s3.put_object(Bucket=BUCKET_NAME, Key="async-input/prompt.csv", Body=async_prompt)
print(f"Uploaded async-input/prompt.csv (1 row).")

# Batch Transform input: same 50K rows.
s3.put_object(Bucket=BUCKET_NAME, Key="batch-input/test_50k.csv", Body=features_only_bytes)
print(f"Uploaded batch-input/test_50k.csv (50000 rows).")

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "sagemaker.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

try:
    iam.create_role(
        RoleName=ENDPOINT_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="labrun mla lab 08 endpoint and batch transform role",
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    print(f"Created role: {ENDPOINT_ROLE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        print(f"Role {ENDPOINT_ROLE_NAME} already exists, reusing.")
    else:
        raise

inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BucketRW",
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject", "s3:DeleteObject"],
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/*",
        },
        {
            "Sid": "BucketList",
            "Effect": "Allow",
            "Action": "s3:ListBucket",
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}",
        },
        {
            "Sid": "CloudWatchLogs",
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents",
            ],
            "Resource": "*",
        },
    ],
}
iam.put_role_policy(
    RoleName=ENDPOINT_ROLE_NAME,
    PolicyName=INLINE_POLICY_NAME,
    PolicyDocument=json.dumps(inline_policy),
)
print(f"Attached inline policy {INLINE_POLICY_NAME}.")

print("Your IAM role is stuck in traffic, give it 10 seconds...")
time.sleep(10)

XGBOOST_IMAGE_URI = sagemaker.image_uris.retrieve(
    framework="xgboost", region=REGION, version="1.5-1"
)
print(f"XGBoost image URI: {XGBOOST_IMAGE_URI}")

# Train once.
sm.create_training_job(
    TrainingJobName=TRAINING_JOB,
    AlgorithmSpecification={
        "TrainingImage": XGBOOST_IMAGE_URI,
        "TrainingInputMode": "File",
    },
    RoleArn=ENDPOINT_ROLE_ARN,
    InputDataConfig=[
        {
            "ChannelName": "train",
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": f"s3://{BUCKET_NAME}/input/train.csv",
                    "S3DataDistributionType": "FullyReplicated",
                }
            },
            "ContentType": "text/csv",
        },
        {
            "ChannelName": "validation",
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": f"s3://{BUCKET_NAME}/input/validation.csv",
                    "S3DataDistributionType": "FullyReplicated",
                }
            },
            "ContentType": "text/csv",
        },
    ],
    OutputDataConfig={"S3OutputPath": f"s3://{BUCKET_NAME}/output/train/"},
    ResourceConfig={"InstanceType": "ml.m5.large", "InstanceCount": 1, "VolumeSizeInGB": 10},
    StoppingCondition={"MaxRuntimeInSeconds": 900},
    HyperParameters={
        "objective": "binary:logistic",
        "num_round": "50",
        "max_depth": "5",
        "eta": "0.2",
        "eval_metric": "auc",
    },
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

print("Training the shared model, give it about 5 minutes...")
elapsed = 0
while elapsed < 900:
    resp = sm.describe_training_job(TrainingJobName=TRAINING_JOB)
    status = resp["TrainingJobStatus"]
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(15)
    elapsed += 15
    if elapsed % 60 == 0:
        print(f"  {elapsed}s elapsed, status: {status}")

if status != "Completed":
    print(f"Training ended with status {status}. Inspect the SageMaker console.")
    raise SystemExit(1)

MODEL_ARTIFACT = sm.describe_training_job(TrainingJobName=TRAINING_JOB)["ModelArtifacts"][
    "S3ModelArtifacts"
]
print(f"Model artifact: {MODEL_ARTIFACT}")

# Build the shared SageMaker Model.
sm.create_model(
    ModelName=MODEL_NAME,
    ExecutionRoleArn=ENDPOINT_ROLE_ARN,
    PrimaryContainer={
        "Image": XGBOOST_IMAGE_URI,
        "ModelDataUrl": MODEL_ARTIFACT,
    },
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
print(f"Created SageMaker Model: {MODEL_NAME}")

# YOUR CODE: Create the Real-Time endpoint config using sm.create_endpoint_config()
# with EndpointConfigName=RT_CONFIG and ProductionVariants=[{
#     "VariantName": "AllTraffic", "ModelName": MODEL_NAME,
#     "InitialInstanceCount": 1, "InstanceType": "ml.t2.medium",
# }], Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
print(f"Created Real-Time endpoint config: {RT_CONFIG}")

# YOUR CODE: Create the Real-Time endpoint using sm.create_endpoint() with
# EndpointName=RT_ENDPOINT, EndpointConfigName=RT_CONFIG,
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
print(f"Creating Real-Time endpoint: {RT_ENDPOINT}")

print("Real-Time endpoint is provisioning ml.t2.medium, give it about 5 minutes...")
elapsed = 0
while elapsed < 900:
    resp = sm.describe_endpoint(EndpointName=RT_ENDPOINT)
    status = resp["EndpointStatus"]
    if status in ("InService", "Failed"):
        break
    time.sleep(15)
    elapsed += 15
    if elapsed % 60 == 0:
        print(f"  {elapsed}s elapsed, status: {status}")

if status != "InService":
    print(f"Real-Time endpoint reached status {status}, not InService.")
    raise SystemExit(1)
print(f"Real-Time endpoint {RT_ENDPOINT} is InService.")

# 100 sequential invocations; exclude first 5 (warm-up).
sample_row = ",".join(["0.1"] * 12)
print("Probing Real-Time endpoint with 100 sequential invocations...")
latencies_ms = []
for i in range(100):
    t0 = time.perf_counter()
    sm_runtime.invoke_endpoint(
        EndpointName=RT_ENDPOINT,
        ContentType="text/csv",
        Body=sample_row.encode("utf-8"),
    )["Body"].read()
    t1 = time.perf_counter()
    latencies_ms.append((t1 - t0) * 1000.0)

warm = sorted(latencies_ms[5:])
RT_P95_MS = warm[int(0.95 * len(warm)) - 1]

# Cost-per-1K calculation. RT bills $0.07/hour continuously. Throughput =
# 1000 / mean_latency_seconds = invocations/sec. Cost-per-1K = $0.07/hour /
# 3600 sec/hour / throughput. The lab approximates throughput as
# 1 / mean_warm_latency since invocations are sequential.
mean_warm_s = statistics.mean(latencies_ms[5:]) / 1000.0
RT_COST_PER_1K = (0.07 / 3600.0) / (1.0 / mean_warm_s) * 1000.0

print(f"Real-Time p95 latency (warm, 95 calls): {RT_P95_MS:.1f} ms")
print(f"Real-Time cost-per-1K-predictions: ${RT_COST_PER_1K:.5f}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Real-Time endpoint is InService on ml.t2.medium, returns
# predictions within p95 < 400 ms (warm calls only).

def checkpoint_1(session):
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            ep = sm_client.describe_endpoint(EndpointName=RT_ENDPOINT)
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Real-Time endpoint {RT_ENDPOINT} was not found. Create it "
                        f"before running the checkpoint."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        if ep.get("EndpointStatus") != "InService":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Real-Time endpoint {RT_ENDPOINT} status is "
                    f"{ep.get('EndpointStatus')!r}, not InService."
                ),
            )

        cfg_name = ep.get("EndpointConfigName")
        cfg = sm_client.describe_endpoint_config(EndpointConfigName=cfg_name)
        variants = cfg.get("ProductionVariants", [])
        if not variants:
            return CheckpointResult(
                status="fail",
                error_reason=f"Endpoint config {cfg_name} has no ProductionVariants.",
            )
        instance_type = variants[0].get("InstanceType")
        if instance_type != "ml.t2.medium":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Real-Time endpoint InstanceType is {instance_type!r}, expected "
                    f"ml.t2.medium. The lab pins this for the cost comparison."
                ),
            )

        if RT_P95_MS is None:
            return CheckpointResult(
                status="fail",
                error_reason="RT_P95_MS is None. Run the 100-invocation probe.",
            )
        if RT_P95_MS > 1000:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Real-Time p95 latency = {RT_P95_MS:.1f} ms is unusually high. "
                    f"Re-run the probe; ml.t2.medium with XGBoost should land under "
                    f"400 ms on warm calls."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Three blanks: create the bucket, create the Real-Time endpoint config, create the endpoint itself. The model build, training, and 100-invocation probe are already wired up. The checkpoint message tells you whether the endpoint is missing, has the wrong instance type, or whether the warm-call p95 looks unhealthy.

</details>

<details><summary>Hint 2 (direction)</summary>

In us-east-1 use `s3.create_bucket(Bucket=BUCKET_NAME)`. The Real-Time endpoint config uses `ProductionVariants[0]` with `InstanceType="ml.t2.medium"`, `InitialInstanceCount=1`, and `ModelName=MODEL_NAME`. The endpoint itself is `sm.create_endpoint(EndpointName=RT_ENDPOINT, EndpointConfigName=RT_CONFIG, Tags=[...])`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1 (the three blocks you need to fill in):

```python
s3.create_bucket(Bucket=BUCKET_NAME)

sm.create_endpoint_config(
    EndpointConfigName=RT_CONFIG,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": MODEL_NAME,
            "InitialInstanceCount": 1,
            "InstanceType": "ml.t2.medium",
        }
    ],
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

sm.create_endpoint(
    EndpointName=RT_ENDPOINT,
    EndpointConfigName=RT_CONFIG,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

</details>

**Wallet check.** Real-Time endpoint on ml.t2.medium at $0.07 per hour starts billing the moment it goes InService and stops when you delete it. The first probe runs in under 30 seconds so you have spent fractions of a penny on the latency probe itself. Training costs about a penny on ml.m5.large for 5 minutes. The endpoint will keep accruing $0.07/hour until cleanup; the cleanup cell at the bottom kills it within 30 minutes. Damage so far: about $0.04. Your coffee still wins.

## Task 2: Deploy the same model to a Serverless Inference endpoint and capture cold + warm latencies

Serverless Inference is $0 idle. It scales automatically and bills per-invocation. The trade-off is cold-start: the first invocation after the endpoint sits idle can take 2-5 seconds while the underlying compute warms. Subsequent invocations are 100-300 ms.

Build it in this order:

1. Create a Serverless endpoint config with `ProductionVariants[0].ServerlessConfig={"MemorySizeInMB": 2048, "MaxConcurrency": 20}`.
2. Create the endpoint and wait for `InService` (~90 seconds).
3. Run 100 sequential invocations. Capture two separate p95s: cold p95 (first 5 calls), and warm p95 (calls 6-100).

The lab tracks cold and warm separately because the cold/warm spread is the lesson. A workload with bursty traffic where cold-starts dominate may prefer real-time; a workload with sustained traffic that keeps the endpoint warm gets serverless's $0 idle benefit without the cold-start penalty.

In [ ]:
# NBVAL_SKIP
# Task 2: Serverless endpoint, with separate cold and warm p95 measurements.

# YOUR CODE: Create the Serverless endpoint config using sm.create_endpoint_config()
# with EndpointConfigName=SL_CONFIG and ProductionVariants=[{
#     "VariantName": "AllTraffic", "ModelName": MODEL_NAME,
#     "ServerlessConfig": {"MemorySizeInMB": 2048, "MaxConcurrency": 20},
# }], Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
print(f"Created Serverless endpoint config: {SL_CONFIG}")

# YOUR CODE: Create the Serverless endpoint using sm.create_endpoint() with
# EndpointName=SL_ENDPOINT, EndpointConfigName=SL_CONFIG,
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
print(f"Creating Serverless endpoint: {SL_ENDPOINT}")

print("Serverless endpoint is warming up, give it about 90 seconds...")
elapsed = 0
while elapsed < 300:
    resp = sm.describe_endpoint(EndpointName=SL_ENDPOINT)
    status = resp["EndpointStatus"]
    if status in ("InService", "Failed"):
        break
    time.sleep(10)
    elapsed += 10
    if elapsed % 30 == 0:
        print(f"  {elapsed}s elapsed, status: {status}")

if status != "InService":
    print(f"Serverless endpoint reached status {status}, not InService.")
    raise SystemExit(1)
print(f"Serverless endpoint {SL_ENDPOINT} is InService.")

sample_row = ",".join(["0.1"] * 12)
print("Probing Serverless endpoint with 100 sequential invocations...")
sl_latencies_ms = []
for i in range(100):
    t0 = time.perf_counter()
    sm_runtime.invoke_endpoint(
        EndpointName=SL_ENDPOINT,
        ContentType="text/csv",
        Body=sample_row.encode("utf-8"),
    )["Body"].read()
    t1 = time.perf_counter()
    sl_latencies_ms.append((t1 - t0) * 1000.0)

cold = sorted(sl_latencies_ms[:5])
warm = sorted(sl_latencies_ms[5:])
SL_COLD_P95_MS = cold[-1] if cold else None
SL_WARM_P95_MS = warm[int(0.95 * len(warm)) - 1]

# Serverless cost: $0.20 / 1M requests + $0.0000200 / GB-second. Per 1K
# invocations: $0.0002 in request fees plus GB-sec at 2 GB and mean warm
# duration.
mean_warm_s = statistics.mean(sl_latencies_ms[5:]) / 1000.0
SL_COST_PER_1K = 0.0002 + (0.0000200 * 2.0 * mean_warm_s * 1000.0)

print(f"Serverless cold p95 latency (first 5 calls): {SL_COLD_P95_MS:.1f} ms")
print(f"Serverless warm p95 latency (calls 6-100):  {SL_WARM_P95_MS:.1f} ms")
print(f"Serverless cost-per-1K-predictions:         ${SL_COST_PER_1K:.5f}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Serverless endpoint is InService with a ServerlessConfig
# block (NOT an InstanceType), and both cold and warm p95s are captured.

def checkpoint_2(session):
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            ep = sm_client.describe_endpoint(EndpointName=SL_ENDPOINT)
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Serverless endpoint {SL_ENDPOINT} was not found. Create it "
                        f"before running the checkpoint."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        if ep.get("EndpointStatus") != "InService":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Serverless endpoint {SL_ENDPOINT} status is "
                    f"{ep.get('EndpointStatus')!r}, not InService."
                ),
            )

        cfg_name = ep.get("EndpointConfigName")
        cfg = sm_client.describe_endpoint_config(EndpointConfigName=cfg_name)
        variants = cfg.get("ProductionVariants", [])
        if not variants or "ServerlessConfig" not in variants[0]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Endpoint config {cfg_name} ProductionVariants[0] has no "
                    f"ServerlessConfig. Recreate with a Serverless block."
                ),
            )

        if SL_COLD_P95_MS is None or SL_WARM_P95_MS is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"SL_COLD_P95_MS={SL_COLD_P95_MS}, SL_WARM_P95_MS={SL_WARM_P95_MS}. "
                    f"Run the 100-invocation probe."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Two blanks: create the Serverless config, create the endpoint. The 100-invocation probe and the cold/warm split are already wired up. The checkpoint message tells you whether the config is missing the Serverless block (a common mistake is leaving InstanceType set, which silently makes the endpoint real-time).

</details>

<details><summary>Hint 2 (direction)</summary>

A Serverless config has `ProductionVariants[0].ServerlessConfig={"MemorySizeInMB": 2048, "MaxConcurrency": 20}`. Do NOT set `InstanceType` or `InitialInstanceCount` for a Serverless config; those are real-time-only fields and including them flips the endpoint type.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2 (the two blocks you need to fill in):

```python
sm.create_endpoint_config(
    EndpointConfigName=SL_CONFIG,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": MODEL_NAME,
            "ServerlessConfig": {"MemorySizeInMB": 2048, "MaxConcurrency": 20},
        }
    ],
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

sm.create_endpoint(
    EndpointName=SL_ENDPOINT,
    EndpointConfigName=SL_CONFIG,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

</details>

**Wallet check.** Serverless is $0 idle and bills $0.20 per 1M requests plus $0.0000200 per GB-second. 100 invocations on a 2048 MB endpoint at ~200 ms each is fractions of a penny. The real-time endpoint is still ticking at $0.07/hour. Damage so far: about $0.05. Your coffee continues to outperform this lab.

## Task 3: Deploy the same model to an Async Inference endpoint and prove S3-input-to-S3-output round-trip

Async Inference is the endpoint type AWS recommends for inferences that exceed the Real-Time 6 MB payload limit or take longer than the 60-second Real-Time max-response. The client posts an S3 input URI, gets an InferenceId back immediately, and polls (or subscribes to SNS) for the output to land in S3.

Build it in this order:

1. Create an Async endpoint config with `AsyncInferenceConfig.OutputConfig.S3OutputPath=s3://{bucket}/async-output/` and `ProductionVariants[0].InstanceType="ml.t2.medium"`.
2. Create the endpoint and wait for `InService` (~5 minutes).
3. Call `sm_runtime.invoke_endpoint_async(EndpointName=..., InputLocation=s3://{bucket}/async-input/prompt.csv)`. Capture the `OutputLocation` from the response.
4. Poll `s3://{bucket}/async-output/...` until the prediction JSON lands (typically 5-30 seconds).
5. Capture `ASYNC_LATENCY_S` from request-time to output-landed-time.

`AsyncInferenceConfig` is the required field. Omitting it silently makes the endpoint real-time. The checkpoint asserts this explicitly.

In [ ]:
# NBVAL_SKIP
# Task 3: Async endpoint with S3-input, S3-output. Code phase plus observe
# phase (poll for the output object).

# YOUR CODE: Create the Async endpoint config using sm.create_endpoint_config()
# with EndpointConfigName=ASYNC_CONFIG, ProductionVariants=[{
#     "VariantName": "AllTraffic", "ModelName": MODEL_NAME,
#     "InitialInstanceCount": 1, "InstanceType": "ml.t2.medium",
# }], AsyncInferenceConfig={
#     "OutputConfig": {"S3OutputPath": f"s3://{BUCKET_NAME}/async-output/"},
# }, Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
print(f"Created Async endpoint config: {ASYNC_CONFIG}")

# YOUR CODE: Create the Async endpoint using sm.create_endpoint() with
# EndpointName=ASYNC_ENDPOINT, EndpointConfigName=ASYNC_CONFIG,
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
print(f"Creating Async endpoint: {ASYNC_ENDPOINT}")

print("Async endpoint is provisioning ml.t2.medium, give it about 5 minutes...")
elapsed = 0
while elapsed < 900:
    resp = sm.describe_endpoint(EndpointName=ASYNC_ENDPOINT)
    status = resp["EndpointStatus"]
    if status in ("InService", "Failed"):
        break
    time.sleep(15)
    elapsed += 15
    if elapsed % 60 == 0:
        print(f"  {elapsed}s elapsed, status: {status}")

if status != "InService":
    print(f"Async endpoint reached status {status}, not InService.")
    raise SystemExit(1)
print(f"Async endpoint {ASYNC_ENDPOINT} is InService.")

print("Submitting async inference against s3://{0}/async-input/prompt.csv".format(BUCKET_NAME))
t0 = time.perf_counter()
async_resp = sm_runtime.invoke_endpoint_async(
    EndpointName=ASYNC_ENDPOINT,
    InputLocation=f"s3://{BUCKET_NAME}/async-input/prompt.csv",
    ContentType="text/csv",
)
inference_id = async_resp.get("InferenceId")
output_location = async_resp.get("OutputLocation")
print(f"  InferenceId: {inference_id}")
print(f"  OutputLocation: {output_location}")

# Poll the output location until the prediction JSON shows up.
print("Polling for async output to land in S3...")
output_bucket = output_location.replace("s3://", "").split("/", 1)[0]
output_key = output_location.replace("s3://", "").split("/", 1)[1]
elapsed = 0
output_found = False
while elapsed < 120:
    try:
        s3.head_object(Bucket=output_bucket, Key=output_key)
        output_found = True
        break
    except ClientError:
        time.sleep(3)
        elapsed += 3

t1 = time.perf_counter()
ASYNC_LATENCY_S = t1 - t0

if not output_found:
    print(f"Async output did not appear within 2 minutes. Check {output_location}.")
    raise SystemExit(1)

# Read the prediction.
output_obj = s3.get_object(Bucket=output_bucket, Key=output_key)
prediction = output_obj["Body"].read().decode("utf-8").strip()
print(f"Async prediction (1 row): {prediction}")
print(f"Async total latency (submit to output landed): {ASYNC_LATENCY_S:.2f} s")

# Async cost: ml.t2.medium at $0.07/hour while up. Per 1K invocations at the
# observed throughput (one prediction at a time, ASYNC_LATENCY_S seconds).
ASYNC_COST_PER_1K = (0.07 / 3600.0) * ASYNC_LATENCY_S * 1000.0
print(f"Async cost-per-1K-predictions (at observed throughput): ${ASYNC_COST_PER_1K:.5f}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Async endpoint is InService, AsyncInferenceConfig is set, and
# the round-trip produced an output S3 object.

def checkpoint_3(session):
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            ep = sm_client.describe_endpoint(EndpointName=ASYNC_ENDPOINT)
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Async endpoint {ASYNC_ENDPOINT} was not found."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        if ep.get("EndpointStatus") != "InService":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Async endpoint {ASYNC_ENDPOINT} status is "
                    f"{ep.get('EndpointStatus')!r}, not InService."
                ),
            )

        cfg_name = ep.get("EndpointConfigName")
        cfg = sm_client.describe_endpoint_config(EndpointConfigName=cfg_name)
        if "AsyncInferenceConfig" not in cfg:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Endpoint config {cfg_name} has no AsyncInferenceConfig. The "
                    f"endpoint silently falls back to Real-Time without it. Recreate "
                    f"with AsyncInferenceConfig.OutputConfig.S3OutputPath set."
                ),
            )

        if ASYNC_LATENCY_S is None:
            return CheckpointResult(
                status="fail",
                error_reason="ASYNC_LATENCY_S is None. Run the async invocation.",
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Two blanks: create the Async config (must include `AsyncInferenceConfig`), create the endpoint. The invocation and poll loop are already wired. The checkpoint message tells you whether the config is missing the `AsyncInferenceConfig` field (a silent fall-back to real-time) or whether the round-trip never completed.

</details>

<details><summary>Hint 2 (direction)</summary>

Async config has BOTH `ProductionVariants[0].InstanceType="ml.t2.medium"` AND `AsyncInferenceConfig={"OutputConfig": {"S3OutputPath": ...}}`. Both fields are required; without `AsyncInferenceConfig` the endpoint type silently becomes Real-Time.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3 (the two blocks you need to fill in):

```python
sm.create_endpoint_config(
    EndpointConfigName=ASYNC_CONFIG,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": MODEL_NAME,
            "InitialInstanceCount": 1,
            "InstanceType": "ml.t2.medium",
        }
    ],
    AsyncInferenceConfig={
        "OutputConfig": {"S3OutputPath": f"s3://{BUCKET_NAME}/async-output/"},
    },
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

sm.create_endpoint(
    EndpointName=ASYNC_ENDPOINT,
    EndpointConfigName=ASYNC_CONFIG,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

</details>

**Wallet check.** Async endpoint on ml.t2.medium at $0.07 per hour, identical to the Real-Time line item. Both are still running and billing in parallel. The cleanup cell tears both down within 30 minutes; if you walk away, both bill $1.68 per day each. Damage so far: about $0.06. The two hourly endpoints are the lab's largest cost surface; the cleanup ordering is critical-first specifically because of them.

## Task 4: Run a Batch Transform job against 50K predictions and capture wall-clock

Batch Transform is the AWS-recommended pattern for one-off bulk inference against a fixed dataset. It scales to the number of workers configured, processes the input prefix, writes outputs back to S3, and shuts down. No persistent endpoint; no idle billing.

Build it in this order:

1. Submit `sm.create_transform_job` with `ModelName=MODEL_NAME`, input at `s3://{bucket}/batch-input/`, output at `s3://{bucket}/batch-output/`, instance type `ml.m5.large`, instance count 1, `MaxConcurrentTransforms=4`, `MaxPayloadInMB=6`, `ContentType="text/csv"`, `SplitType="Line"`.
2. Wait for `TransformJobStatus=Completed` (~5-10 minutes for 50K predictions on ml.m5.large).
3. Confirm `list_objects_v2` against the batch-output prefix returns at least one output object.
4. Capture wall-clock time into `BATCH_WALLCLOCK_MIN` and per-1K cost into `BATCH_COST_PER_1K` using `BillableTimeInSeconds`.

Batch Transform's value: no idle billing, native S3 integration, per-record output. It is NOT a streaming or real-time pattern; it does not respond to per-request invocations.

In [ ]:
# NBVAL_SKIP
# Task 4: Batch Transform job over 50K predictions.

# YOUR CODE: Submit the Batch Transform job using sm.create_transform_job()
# with TransformJobName=BATCH_JOB, ModelName=MODEL_NAME,
# TransformInput={
#     "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix",
#         "S3Uri": f"s3://{BUCKET_NAME}/batch-input/"}},
#     "ContentType": "text/csv", "SplitType": "Line",
# },
# TransformOutput={
#     "S3OutputPath": f"s3://{BUCKET_NAME}/batch-output/",
#     "AssembleWith": "Line",
# },
# TransformResources={"InstanceType": "ml.m5.large", "InstanceCount": 1},
# MaxConcurrentTransforms=4, MaxPayloadInMB=6,
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
print(f"Submitted Batch Transform job: {BATCH_JOB}")

print("Batch Transform is chewing through 50K predictions, give it about 10 minutes...")
elapsed = 0
while elapsed < 1500:
    resp = sm.describe_transform_job(TransformJobName=BATCH_JOB)
    status = resp["TransformJobStatus"]
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(15)
    elapsed += 15
    if elapsed % 60 == 0:
        print(f"  {elapsed}s elapsed, status: {status}")

if status != "Completed":
    print(f"Batch Transform reached status {status}, not Completed.")
    raise SystemExit(1)

resp = sm.describe_transform_job(TransformJobName=BATCH_JOB)
billable_seconds = resp.get("BillableTimeInSeconds", elapsed)
BATCH_WALLCLOCK_MIN = elapsed / 60.0
# Batch Transform cost: ml.m5.large at $0.115/hour for billable seconds.
batch_total_cost = (0.115 / 3600.0) * billable_seconds
BATCH_COST_PER_1K = batch_total_cost / 50.0  # 50K predictions, so per-1K is total/50

# Confirm output objects exist.
outputs = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix="batch-output/")
print(f"Batch Transform completed in {BATCH_WALLCLOCK_MIN:.1f} minutes.")
print(f"  Output objects: {outputs.get('KeyCount', 0)}")
print(f"  Billable seconds: {billable_seconds}")
print(f"  Total cost: ${batch_total_cost:.5f}")
print(f"  Cost-per-1K-predictions: ${BATCH_COST_PER_1K:.5f}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Batch Transform job is Completed and at least one output
# object lives under batch-output/.

def checkpoint_4(session):
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            job = sm_client.describe_transform_job(TransformJobName=BATCH_JOB)
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Batch Transform job {BATCH_JOB} was not found. Submit the "
                        f"job before running the checkpoint."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        if job["TransformJobStatus"] != "Completed":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Batch Transform {BATCH_JOB} status is "
                    f"{job['TransformJobStatus']!r}, not Completed."
                ),
            )

        instance_type = job.get("TransformResources", {}).get("InstanceType")
        if instance_type != "ml.m5.large":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Batch Transform InstanceType is {instance_type!r}, expected "
                    f"ml.m5.large for the cost comparison."
                ),
            )

        outputs = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix="batch-output/")
        if outputs.get("KeyCount", 0) == 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Batch Transform reported Completed but batch-output/ contains "
                    f"no objects. Check CloudWatch logs for the job."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

One blank: the `create_transform_job` call. The poll loop, output check, and cost math are already wired up. The checkpoint message tells you whether the job is missing, the instance type is wrong, or the output prefix is empty.

</details>

<details><summary>Hint 2 (direction)</summary>

`sm.create_transform_job(TransformJobName=..., ModelName=..., TransformInput=..., TransformOutput=..., TransformResources=..., MaxConcurrentTransforms=4, MaxPayloadInMB=6, Tags=[...])`. `TransformInput.SplitType="Line"` is what makes XGBoost see one CSV row per inference. `TransformOutput.AssembleWith="Line"` writes one prediction per line back to S3.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4 (the one block you need to fill in):

```python
sm.create_transform_job(
    TransformJobName=BATCH_JOB,
    ModelName=MODEL_NAME,
    TransformInput={
        "DataSource": {
            "S3DataSource": {
                "S3DataType": "S3Prefix",
                "S3Uri": f"s3://{BUCKET_NAME}/batch-input/",
            }
        },
        "ContentType": "text/csv",
        "SplitType": "Line",
    },
    TransformOutput={
        "S3OutputPath": f"s3://{BUCKET_NAME}/batch-output/",
        "AssembleWith": "Line",
    },
    TransformResources={"InstanceType": "ml.m5.large", "InstanceCount": 1},
    MaxConcurrentTransforms=4,
    MaxPayloadInMB=6,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

</details>

**Wallet check.** Batch Transform on ml.m5.large for ~10 minutes lands at about two cents. Unlike the real-time and async endpoints, batch shuts down on its own when the job finishes; there is no idle window. Damage so far: about $0.09. The two hourly endpoints are still ticking; the cleanup cell at the bottom kills them.

## Task 5: Build the four-architecture comparison and surface the metric

All four endpoint types are stood up and measured. Build the side-by-side table the product manager wants:

- Real-Time: instance type, warm p95, cost-per-1K (at observed throughput).
- Serverless: cold p95, warm p95, cost-per-1K.
- Async: end-to-end latency, cost-per-1K (at observed throughput).
- Batch Transform: wall-clock for 50K predictions, cost-per-1K.

Checkpoint 5 captures these into a `comparisonMetric` string that the Option D Colab card surfaces on PASS. The validator confirms all measurements are well-formed and re-queries the AWS side to confirm the four endpoint types each have the expected `ProductionVariants` shape.

Per blueprint Section 21, the validator does NOT make a "which type wins" assertion in code. Each of the three workload shapes (100 rps business hours, zero overnight, weekly 50K backfill) maps to a different best-fit endpoint type. The constraint-driven comparison happens in the reflection MCQ.

In [ ]:
# NBVAL_SKIP
# Task 5: Side-by-side comparison print-out. No new AWS calls; reads the
# captured metrics from Tasks 1-4 and surfaces them in one place.

print("Side-by-side comparison across four endpoint types:")
print()
print(f"  Real-Time  (ml.t2.medium):")
print(f"    warm p95: {RT_P95_MS:.1f} ms")
print(f"    cost-per-1K-predictions: ${RT_COST_PER_1K:.5f}")
print(f"    idle billing: $0.07 per hour, always-on")
print()
print(f"  Serverless (2048 MB, MaxConcurrency=20):")
print(f"    cold p95: {SL_COLD_P95_MS:.1f} ms")
print(f"    warm p95: {SL_WARM_P95_MS:.1f} ms")
print(f"    cost-per-1K-predictions: ${SL_COST_PER_1K:.5f}")
print(f"    idle billing: $0 (scales to zero)")
print()
print(f"  Async      (ml.t2.medium, S3-in, S3-out):")
print(f"    end-to-end latency: {ASYNC_LATENCY_S:.2f} s")
print(f"    cost-per-1K-predictions (at observed throughput): ${ASYNC_COST_PER_1K:.5f}")
print(f"    idle billing: $0.07 per hour while up; can scale to 0 with MinCapacity=0")
print()
print(f"  Batch      (ml.m5.large, 50K predictions):")
print(f"    total wall-clock: {BATCH_WALLCLOCK_MIN:.1f} minutes")
print(f"    cost-per-1K-predictions: ${BATCH_COST_PER_1K:.5f}")
print(f"    idle billing: $0 (shuts down when job completes)")
print()
print("Workload shapes from the scenario:")
print(f"  100 rps business hours, zero overnight: Serverless is the canonical fit")
print(f"  Long-running per-request with large payloads: Async")
print(f"  Weekly 50K backfill: Batch Transform")
print(f"  100 rps with strict sub-50 ms p95 latency: Real-Time (the one case)")

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: All per-architecture measurements are captured and well-formed.
# Per blueprint Section 21, the validator does NOT compute a winner; it confirms
# the metrics exist so the Option D card can surface them.

def checkpoint_5(session):
    try:
        captures = {
            "RT_P95_MS": RT_P95_MS,
            "RT_COST_PER_1K": RT_COST_PER_1K,
            "SL_COLD_P95_MS": SL_COLD_P95_MS,
            "SL_WARM_P95_MS": SL_WARM_P95_MS,
            "SL_COST_PER_1K": SL_COST_PER_1K,
            "ASYNC_LATENCY_S": ASYNC_LATENCY_S,
            "ASYNC_COST_PER_1K": ASYNC_COST_PER_1K,
            "BATCH_WALLCLOCK_MIN": BATCH_WALLCLOCK_MIN,
            "BATCH_COST_PER_1K": BATCH_COST_PER_1K,
        }
        for name, val in captures.items():
            if val is None:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{name} is None. Run the previous tasks before running "
                        f"Checkpoint 5; the comparison cell relies on all nine captures."
                    ),
                )
            if val < 0:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"{name} = {val}, must be non-negative.",
                )

        # Sanity-check the four AWS endpoint types still exist.
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        for name in (RT_ENDPOINT, SL_ENDPOINT, ASYNC_ENDPOINT):
            try:
                sm_client.describe_endpoint(EndpointName=name)
            except ClientError as e:
                if e.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"Endpoint {name} no longer exists. The comparison requires "
                            f"all three live endpoints plus the Batch Transform job."
                        ),
                    )

        try:
            batch_job = sm_client.describe_transform_job(TransformJobName=BATCH_JOB)
            if batch_job["TransformJobStatus"] != "Completed":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Batch Transform {BATCH_JOB} status is "
                        f"{batch_job['TransformJobStatus']!r}, not Completed."
                    ),
                )
        except ClientError:
            return CheckpointResult(
                status="fail",
                error_reason=f"Batch Transform {BATCH_JOB} was not found.",
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

Zero blanks. The comparison cell only reads the captured metrics and prints them. If the checkpoint fails, one of the earlier task probes did not run; the fix is upstream.

</details>

<details><summary>Hint 2 (direction)</summary>

If `RT_P95_MS` or `SL_WARM_P95_MS` is None, re-run the 100-invocation probe in Task 1 or Task 2. If `ASYNC_LATENCY_S` is None, re-run Task 3. If `BATCH_WALLCLOCK_MIN` is None, re-run Task 4. The comparison cell does not write to AWS.

</details>

<details><summary>Hint 3 (spoiler)</summary>

There is nothing to fill in. If Checkpoint 5 fails the fix is in whichever upstream task did not capture its metric.

</details>

**Wallet check.** No new compute. The four `describe_*` calls in the checkpoint are free. The real-time and async endpoints are still ticking and need the cleanup cell to come down. Damage so far: still about $0.09. The receipt for the product manager is the cheapest part of this lab.

## Cleanup

Cleanup tears the three endpoints down in critical-first order: Real-Time first (highest hourly cost), then Async (same hourly cost, same urgency), then Serverless ($0 idle so non-critical). Endpoint configs come down after the endpoints (config delete fails while any endpoint references it). The SageMaker Model and the Batch Transform job come down next. Training jobs cannot be deleted; the cleanup defensively stops any in-flight job. Then `run_cleanup` walks the IAM role and S3 bucket.

Re-running the cell is safe. Each manual delete is wrapped in try/except that swallows "already gone" errors.

In [ ]:
# NBVAL_SKIP
# Cleanup. Manual SageMaker teardown first (no adapters in labrun-checks 0.6.0
# for endpoint/model/transform/training), then run_cleanup walks iam_role +
# s3_bucket.

sm_cleanup = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

manual_warnings = []


def _safe_delete(label, action, fallback_cmd):
    try:
        action()
        print(f"Deleted {label}.")
    except ClientError as e:
        if e.response["Error"]["Code"] in (
            "ValidationException",
            "ResourceNotFound",
            "ResourceNotFoundException",
        ):
            print(f"{label} already gone, skipping.")
        else:
            manual_warnings.append(
                f"FAILED TO DELETE: {label}. Error: {e}. Run manually: {fallback_cmd}"
            )


def _wait_for_endpoint_gone(name: str, ceiling: int = 300) -> bool:
    elapsed = 0
    while elapsed < ceiling:
        try:
            sm_cleanup.describe_endpoint(EndpointName=name)
            time.sleep(10)
            elapsed += 10
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
                return True
            raise
    return False


# Stop any in-flight training or transform job.
try:
    resp = sm_cleanup.describe_training_job(TrainingJobName=TRAINING_JOB)
    if resp["TrainingJobStatus"] == "InProgress":
        sm_cleanup.stop_training_job(TrainingJobName=TRAINING_JOB)
        print(f"Requested stop on in-flight training job {TRAINING_JOB}")
except ClientError as e:
    if e.response["Error"]["Code"] not in ("ValidationException", "ResourceNotFound"):
        manual_warnings.append(f"FAILED TO DESCRIBE training_job: {e}")

try:
    resp = sm_cleanup.describe_transform_job(TransformJobName=BATCH_JOB)
    if resp["TransformJobStatus"] == "InProgress":
        sm_cleanup.stop_transform_job(TransformJobName=BATCH_JOB)
        print(f"Requested stop on in-flight transform job {BATCH_JOB}")
except ClientError as e:
    if e.response["Error"]["Code"] not in ("ValidationException", "ResourceNotFound"):
        manual_warnings.append(f"FAILED TO DESCRIBE transform_job: {e}")

# Critical-first endpoint teardown: RT, Async, then Serverless.
endpoints_gone = {"rt": False, "async": False, "sl": False}
for label, name in (("rt", RT_ENDPOINT), ("async", ASYNC_ENDPOINT), ("sl", SL_ENDPOINT)):
    _safe_delete(
        f"endpoint {name}",
        lambda n=name: sm_cleanup.delete_endpoint(EndpointName=n),
        f"aws sagemaker delete-endpoint --endpoint-name {name}",
    )
    print(f"Waiting for endpoint {name} to disappear...")
    if _wait_for_endpoint_gone(name):
        endpoints_gone[label] = True
    else:
        manual_warnings.append(
            f"Endpoint {name} did not disappear within 5 minutes. Run manually: "
            f"aws sagemaker delete-endpoint --endpoint-name {name}"
        )

# Configs (after endpoints).
for cfg_name in (RT_CONFIG, ASYNC_CONFIG, SL_CONFIG):
    _safe_delete(
        f"endpoint config {cfg_name}",
        lambda n=cfg_name: sm_cleanup.delete_endpoint_config(EndpointConfigName=n),
        f"aws sagemaker delete-endpoint-config --endpoint-config-name {cfg_name}",
    )

# Shared Model.
_safe_delete(
    f"model {MODEL_NAME}",
    lambda: sm_cleanup.delete_model(ModelName=MODEL_NAME),
    f"aws sagemaker delete-model --model-name {MODEL_NAME}",
)

# Now hand off to run_cleanup for the IAM role and S3 bucket.
result = run_cleanup(CLEANUP_MANIFEST)

for warning in manual_warnings:
    print(warning)
for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if manual_warnings or result.warnings or result.orphans:
    print()

failed_ids = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

# Accounting:
# critical: Real-Time endpoint, Async endpoint (both $0.07/hour). 2 total.
critical_total = 2
critical_destroyed = (1 if endpoints_gone["rt"] else 0) + (
    1 if endpoints_gone["async"] else 0
)
standard_total = len(CLEANUP_MANIFEST)
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids) + len(manual_warnings)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed} of {critical_total}")
print(f"Standard resources destroyed: {standard_destroyed} of {standard_total}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

final_status = "clean" if (
    failed_count == 0
    and result.status == "clean"
    and endpoints_gone["rt"]
    and endpoints_gone["async"]
) else "dirty"
cleanup(status=final_status)

if failed_count > 0 or not (endpoints_gone["rt"] and endpoints_gone["async"]):
    sys.exit(1)

**Session total: about $0.08 to $0.15.** Real-Time and Async endpoints on ml.t2.medium at $0.07 per hour each for roughly 30 minutes lands at about seven cents combined. Batch Transform on ml.m5.large for ~10 minutes adds about two cents. Serverless invocations are fractions of a penny. The shared training job adds another penny. If you walk away with both hourly endpoints running, the bill is about $3.40 per day; the cleanup cell stops that the moment it runs cleanly. Check your AWS Billing console in 24 hours to confirm zero ongoing charges from this lab.

## Reflection

These are not graded. They are for you.

1. Walk through the four SageMaker endpoint types in terms of three dimensions: idle cost, max payload size, and best-fit workload shape. Real-Time bills continuously and supports under 6 MB payloads with sub-second latency. Serverless is $0 idle and supports under 4 MB payloads with cold-start latency. Async is auto-scaling and supports up to 1 GB payloads with up to 60 minutes processing time. Batch Transform is the right tool for bulk inference against a fixed dataset. For the scenario in this lab (100 rps business hours, zero overnight, weekly 50K backfill), which endpoint type fits which workload, and which type fits NONE of the three?

2. Walk through what changes if the recommendation model jumps from 100 rps to 10,000 rps during business hours. Which endpoint types still fit, which break (and how), and what is the cost-vs-latency trade-off at the higher throughput? At what request volume does Real-Time with auto scaling start to beat Serverless on cost-per-1K, and why?

## Exam-Style Practice

**Question 1 (Domain 3, endpoint type selection by workload shape; constraint-driven comparison per blueprint Section 21):**

A team's recommendation model runs 100 inferences per second during business hours, near zero overnight, and occasionally needs to score 50,000 historical records in a one-off backfill. Which combination of SageMaker endpoint types fits the three workload shapes with the lowest total monthly cost?

A. Real-Time endpoint for business hours, paused overnight via auto scaling, with Batch Transform for the backfill.

B. Serverless Inference endpoint for the bursty real-time traffic (handles business hours and overnight with $0 idle) plus Batch Transform for the 50K backfill.

C. Real-Time endpoint with `MinCapacity=1, MaxCapacity=10` for the real-time traffic plus Async Inference for the backfill.

D. Three Real-Time endpoints (one per workload shape) with manual scaling.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong on "pause overnight via auto scaling": Real-Time auto scaling can scale down to MinCapacity=1 but not 0. The endpoint bills continuously overnight.
- B is correct: Serverless Inference is $0 idle and scales automatically; it handles the bursty 100 rps and zero-overnight pattern with no idle cost. Batch Transform is the AWS-recommended pattern for one-off bulk inference against a fixed dataset; it scales to the number of workers configured and shuts down when the job completes.
- C is partially right: MinCapacity=1 still bills 1 instance continuously overnight, and Async Inference is not the right pattern for a one-off 50K backfill (Async fits long-running per-request inference like document processing; Batch Transform is the bulk-inference pattern).
- D is the most expensive option: three persistent endpoints continuously billing instance-hours, with no per-shape benefit.

</details>

**Question 2 (Domain 3, Async vs Serverless distinction):**

A team is choosing between SageMaker Async Inference and Serverless Inference for a workload that processes 50 MB request payloads with up to 5 minutes of inference time per request. Which fits, and why?

A. Serverless Inference; it scales to 0 and supports payloads up to 50 MB by default.

B. Async Inference; it accepts payloads up to 1 GB via S3 input and supports inference times up to 60 minutes per request.

C. Real-Time endpoint with extended timeout; it handles all payload sizes and inference times.

D. Batch Transform; it processes 50 MB payloads as a stream.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong on payload size: Serverless Inference has a 4 MB payload ceiling per request. 50 MB exceeds the limit and the request fails.
- B is correct: Async Inference accepts inputs via S3 URIs (up to 1 GB), supports up to 60 minutes of inference time per request, and writes outputs back to S3. Both constraints (50 MB payload, 5-minute inference) are well within Async's published limits.
- C is wrong on "all payload sizes": Real-Time endpoints have a 6 MB payload ceiling per request (HTTP body limit). 50 MB exceeds the limit.
- D is wrong: Batch Transform is a job, not an endpoint. It processes a directory of files in batch, not per-request streaming. The Async pattern fits when the team needs per-request response handling at large payload sizes.

</details>